# ============================================================
# Projet 3 - Prédisez la consommation d'énergie des bâtiments
# Viken KHATCHERIAN
# formation AI Engineer - OpenClassrooms
# ============================================================

## Projet complet de modélisation prédictive de la consommation énergétique des bâtiments non destinés à l'habitation de la ville américaine de Seattle.
## Le workflow intègre une analyse exploratoire jusqu'à l’optimisation du modèle, en intégrant des considérations de robustesse statistique et d’interprétabilité.

# Etape 1 : Raisonnement mathématique de formalisation du problème à résoudre

# Etape 2 : Chargement et compréhension des données

In [29]:
# Importation des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
# Voir toutes les colonnes du dataframe après avoir chargé le csv
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [31]:
# Chargement du dataset
df_2016beb = pd.read_csv("2016_Building_Energy_Benchmarking.csv")

In [32]:
# Nombre de lignes et de colonnes du dataframe obtenu
df_2016beb.shape

(3376, 46)

In [33]:
# Aperçu du nom des colonnes
df_2016beb.columns

Index(['OSEBuildingID', 'DataYear', 'BuildingType', 'PrimaryPropertyType',
       'PropertyName', 'Address', 'City', 'State', 'ZipCode',
       'TaxParcelIdentificationNumber', 'CouncilDistrictCode', 'Neighborhood',
       'Latitude', 'Longitude', 'YearBuilt', 'NumberofBuildings',
       'NumberofFloors', 'PropertyGFATotal', 'PropertyGFAParking',
       'PropertyGFABuilding(s)', 'ListOfAllPropertyUseTypes',
       'LargestPropertyUseType', 'LargestPropertyUseTypeGFA',
       'SecondLargestPropertyUseType', 'SecondLargestPropertyUseTypeGFA',
       'ThirdLargestPropertyUseType', 'ThirdLargestPropertyUseTypeGFA',
       'YearsENERGYSTARCertified', 'ENERGYSTARScore', 'SiteEUI(kBtu/sf)',
       'SiteEUIWN(kBtu/sf)', 'SourceEUI(kBtu/sf)', 'SourceEUIWN(kBtu/sf)',
       'SiteEnergyUse(kBtu)', 'SiteEnergyUseWN(kBtu)', 'SteamUse(kBtu)',
       'Electricity(kWh)', 'Electricity(kBtu)', 'NaturalGas(therms)',
       'NaturalGas(kBtu)', 'DefaultData', 'Comments', 'ComplianceStatus',
       'Outlier

In [34]:
# Colonnes contenant la chaîne "EPA"
colonnesEPA = df_2016beb.columns[df_2016beb.columns.str.contains("EPA")]
print(colonnesEPA)

Index([], dtype='str')


## Identification des colonnes pertinentes à garder

## Label
- SiteEnergyUse(kBtu)  

## Structure
- YearBuilt
- NumberofFloors
- NumberofBuildings
- PropertyGFATotal
- PropertyGFABuildings
- PropertyGFAParking

## Usage
- BuildingType
- EPAPropertyType (non présente dans le dataframe)
- LargestPropertyUseType
- LargestPropertyUseTypeGFA
- SecondLargestPropertyUseType
- SecondLargestPropertyUseTypeGFA
- ThirdLargestPropertyUseType
- ThirdLargestPropertyUseTypeGFA

## Localisation
- Neighborhood
- CouncilDistrictCode
- Latitude
- Longitude
- ZipCode

## Temps
- datayear (non conservé car 2016 uniquement)

## Optionnel
- ENERGYSTARScore

In [35]:
# Identification du type des colonnes et du nombre de valeurs manquantes
df_2016beb.info()

<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 46 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    3376 non-null   int64  
 1   DataYear                         3376 non-null   int64  
 2   BuildingType                     3376 non-null   str    
 3   PrimaryPropertyType              3376 non-null   str    
 4   PropertyName                     3376 non-null   str    
 5   Address                          3376 non-null   str    
 6   City                             3376 non-null   str    
 7   State                            3376 non-null   str    
 8   ZipCode                          3360 non-null   float64
 9   TaxParcelIdentificationNumber    3376 non-null   str    
 10  CouncilDistrictCode              3376 non-null   int64  
 11  Neighborhood                     3376 non-null   str    
 12  Latitude                       

In [42]:
# Liste des colonnes retenues (sans EPAPropertyType ni datayear)
cols_pertinentes = [
    # Label
    "SiteEnergyUse(kBtu)",  
    
    # Structure
    "YearBuilt",
    "NumberofFloors",
    "NumberofBuildings",
    "PropertyGFATotal",
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    
    # Usage
    "BuildingType",
    "LargestPropertyUseType",
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseType",
    "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseType",
    "ThirdLargestPropertyUseTypeGFA",
    
    # Localisation
    "Neighborhood",
    "CouncilDistrictCode",
    "Latitude",
    "Longitude",
    "ZipCode",
    
    # Optionnel
    "ENERGYSTARScore"
]

# Sélection du sous-dataframe
df_2016beb_rel =df_2016beb[cols_pertinentes]

# Afficher le nombre de valeurs non-null par colonne
non_null_counts = df_2016beb_rel.notnull().sum().sort_values(ascending=False)
print("Nombre de valeurs non-null par colonne :\n", non_null_counts)

# Pour les colonnes catégorielles : afficher le nombre de valeurs uniques et top valeurs
categorical_cols = [
    "BuildingType",
    "LargestPropertyUseType",
    "SecondLargestPropertyUseType",
    "ThirdLargestPropertyUseType",
    "Neighborhood",
    "CouncilDistrictCode",
    "ZipCode"
]

for col in categorical_cols:
    print(f"\nColonne : {col}")
    print("Nombre de valeurs uniques :", df_2016beb_rel[col].nunique())
    print("Top 5 valeurs les plus fréquentes :\n", df_2016beb_rel[col].value_counts().head())
    
# Pour les colonnes numériques : statistique descriptive rapide
numeric_cols = [
    "YearBuilt",
    "NumberofFloors",
    "NumberofBuildings",
    "PropertyGFATotal",
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseTypeGFA",
    "Latitude",
    "Longitude",
    "ENERGYSTARScore",
    "SiteEnergyUse(kBtu)" 
]

print("\nStatistiques descriptives des colonnes numériques :\n")
print(df_2016beb_rel[numeric_cols].describe())

Nombre de valeurs non-null par colonne :
 YearBuilt                          3376
NumberofFloors                     3376
PropertyGFABuilding(s)             3376
PropertyGFATotal                   3376
Latitude                           3376
Longitude                          3376
PropertyGFAParking                 3376
BuildingType                       3376
Neighborhood                       3376
CouncilDistrictCode                3376
SiteEnergyUse(kBtu)                3371
NumberofBuildings                  3368
ZipCode                            3360
LargestPropertyUseType             3356
LargestPropertyUseTypeGFA          3356
ENERGYSTARScore                    2533
SecondLargestPropertyUseTypeGFA    1679
SecondLargestPropertyUseType       1679
ThirdLargestPropertyUseType         596
ThirdLargestPropertyUseTypeGFA      596
dtype: int64

Colonne : BuildingType
Nombre de valeurs uniques : 8
Top 5 valeurs les plus fréquentes :
 BuildingType
NonResidential          1460
Multifamily 

# Etape 3 : Analyse Exploratoire et Approfondie (EDA)

In [43]:
# Liste des colonnes retenues pour l'EDA
cols_pertinentes = [
    # Label
    "SiteEnergyUse(kBtu)",  
    
    # Structure
    "YearBuilt",
    "NumberofFloors",
    "NumberofBuildings",
    "PropertyGFATotal",
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    
    # Usage
    "BuildingType",
    "LargestPropertyUseType",
    "LargestPropertyUseTypeGFA",
    "SecondLargestPropertyUseType",
    "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseType",
    "ThirdLargestPropertyUseTypeGFA",
    
    # Localisation
    "Neighborhood",
    "CouncilDistrictCode",
    "Latitude",
    "Longitude",
    "ZipCode"
]

# Etape 4 : Feature Engineering

# Etape 5 : Préparation des données

## Etape 5.1 : Définition X et y 

## Etape 5.2 : Gestion des outliers

## Etape 5.3 : Encoding

## Etape 5.4 : Scaling

## Etape 5.5 : Construction du ColumnTransformer

# Etape 6 : Méthodologie de modélisation

# Etape 7 : Comparaison des modèles

# Etape 8 : Optimisation

# Etape 9 : Interprétation du modèle final

# Etape 10 : Limites & perspectives

# Etape 11 : Conclusion